# Metro Manila Traffic Incident Analysis

This project analyzes traffic incident data across Metro Manila using a Python → SQL → Power BI workflow.

The goal is to identify traffic incident patterns such as:
- Cities with the highest number of incidents
- Peak hours when incidents occur most frequently
- Common types of traffic incidents

The results of this analysis are used to create an interactive Power BI dashboard that visualizes congestion hotspots and time-based trends.

**Tools used:** Python (Pandas), SQL (SQLite), Power BI

## Step 1: Load the Dataset

In this step, we load the raw traffic incident dataset into a pandas DataFrame.  
This allows us to inspect the structure of the data and understand what variables are available for analysis.

In [1]:
import pandas as pd

df = pd.read_csv("data_mmda_traffic_spatial.csv")

df.head()

,Date,Time,City,Location,Latitude,Longitude,High_Accuracy,Direction,Type,Lanes_Blocked,Involved,Tweet,Source
0,2018-08-20,7:55 AM,Pasig City,ORTIGAS EMERALD,14.586343,121.061481,1,EB,VEHICULAR ACCIDENT,1.0,TAXI AND MC,MMDA ALERT: Vehicular accident at Ortigas Emer...,https://twitter.com/mmda/status/10313302019705...
1,2018-08-20,8:42 AM,Mandaluyong,EDSA GUADIX,14.589432,121.057243,1,NB,STALLED L300 DUE TO MECHANICAL PROBLEM,1.0,L300,MMDA ALERT: Stalled L300 due to mechanical pro...,https://twitter.com/mmda/status/10313462477459...
2,2018-08-20,9:13 AM,Makati City,EDSA ROCKWELL,14.559818,121.040737,1,SB,VEHICULAR ACCIDENT,1.0,SUV AND L300,MMDA ALERT: Vehicular accident at EDSA Rockwel...,https://twitter.com/mmda/status/10313589669896...
3,2018-08-20,8:42 AM,Mandaluyong,EDSA GUADIX,14.589432,121.057243,1,NB,STALLED L300 DUE TO MECHANICAL PROBLEM,1.0,L300,MMDA ALERT: Stalled L300 due to mechanical pro...,https://twitter.com/mmda/status/10313590696535...
4,2018-08-20,10:27 AM,San Juan,ORTIGAS CLUB FILIPINO,14.601846,121.046754,1,EB,VEHICULAR ACCIDENT,1.0,2 CARS,MMDA ALERT: Vehicular accident at Ortigas Club...,https://twitter.com/mmda/status/10313711248424...


## Step 2: Standardize Column Names

To make the dataset easier to work with, we standardize the column names by converting them to lowercase.  
This prevents errors caused by inconsistent capitalization when referencing columns later in the analysis.

In [2]:
# clean column names
df.columns = df.columns.str.lower()

df.head()

,date,time,city,location,latitude,longitude,high_accuracy,direction,type,lanes_blocked,involved,tweet,source
0,2018-08-20,7:55 AM,Pasig City,ORTIGAS EMERALD,14.586343,121.061481,1,EB,VEHICULAR ACCIDENT,1.0,TAXI AND MC,MMDA ALERT: Vehicular accident at Ortigas Emer...,https://twitter.com/mmda/status/10313302019705...
1,2018-08-20,8:42 AM,Mandaluyong,EDSA GUADIX,14.589432,121.057243,1,NB,STALLED L300 DUE TO MECHANICAL PROBLEM,1.0,L300,MMDA ALERT: Stalled L300 due to mechanical pro...,https://twitter.com/mmda/status/10313462477459...
2,2018-08-20,9:13 AM,Makati City,EDSA ROCKWELL,14.559818,121.040737,1,SB,VEHICULAR ACCIDENT,1.0,SUV AND L300,MMDA ALERT: Vehicular accident at EDSA Rockwel...,https://twitter.com/mmda/status/10313589669896...
3,2018-08-20,8:42 AM,Mandaluyong,EDSA GUADIX,14.589432,121.057243,1,NB,STALLED L300 DUE TO MECHANICAL PROBLEM,1.0,L300,MMDA ALERT: Stalled L300 due to mechanical pro...,https://twitter.com/mmda/status/10313590696535...
4,2018-08-20,10:27 AM,San Juan,ORTIGAS CLUB FILIPINO,14.601846,121.046754,1,EB,VEHICULAR ACCIDENT,1.0,2 CARS,MMDA ALERT: Vehicular accident at Ortigas Club...,https://twitter.com/mmda/status/10313711248424...


## Step 3: Combine Date and Time

The dataset stores the date and time in separate columns.  
In this step, we combine them into a single datetime column to make time-based analysis possible.

We also extract the **hour of the day**, which will allow us to analyze traffic incident patterns during rush hours.

In [7]:
# Combine date + time into one string
df['datetime_str'] = df['date'] + ' ' + df['time']

# Convert using explicit format to handle AM/PM
df['datetime'] = pd.to_datetime(df['datetime_str'], format='%Y-%m-%d %I:%M %p', errors='coerce')

# Extract hour (24-hour)
df['hour'] = df['datetime'].dt.hour

# Check
df[['datetime','hour']].head()

,datetime,hour
0,2018-08-20 07:55:00,7.0
1,2018-08-20 08:42:00,8.0
2,2018-08-20 09:13:00,9.0
3,2018-08-20 08:42:00,8.0
4,2018-08-20 10:27:00,10.0


## Step 4: Analyze Incidents by City

Here we calculate the number of traffic incidents reported in each city.  
This helps identify which areas experience the highest concentration of traffic disruptions.

In [8]:
accidents_by_city = df.groupby("city").size().reset_index(name="accidents")

accidents_by_city = accidents_by_city.sort_values("accidents", ascending=False)

accidents_by_city.head(10)

,city,accidents
11,Quezon City,8725
3,Mandaluyong,2978
1,Makati City,2431
10,Pasig City,1723
9,Pasay City,445
4,Manila,354
12,San Juan,121
5,Marikina,119
7,ParaÃ±aque,107
0,Kalookan City,70


## Step 5: Analyze Incidents by Hour

Traffic incidents often follow daily traffic patterns.  
In this step, we group incidents by hour to identify peak congestion periods.

In [9]:
accidents_by_hour = df.groupby("hour").size().reset_index(name="accidents")

accidents_by_hour

,hour,accidents
0,0.0,247
1,1.0,119
2,2.0,55
3,3.0,67
4,4.0,287
5,5.0,606
6,6.0,1013
7,7.0,1354
8,8.0,1271
9,9.0,1192


## Step 6: Analyze Types of Traffic Incidents

Different types of incidents may contribute to congestion differently.  
This step identifies the most common incident types recorded in the dataset.

In [10]:
incident_types = df.groupby("type").size().reset_index(name="count")

incident_types = incident_types.sort_values("count", ascending=False)

incident_types

,type,count
471,VEHICULAR ACCIDENT,11775
247,STALLED BUS DUE TO MECHANICAL PROBLEM,864
95,MULTIPLE COLLISION,623
268,STALLED CAR DUE TO MECHANICAL PROBLEM,593
417,STALLED TRUCK DUE TO MECHANICAL PROBLEM,395
...,...,...
204,STALLED 10 WHEELER TRUCK DUE TO EMPTY GAS,1
203,STALLED 10 WHEELER TRUCK DUE MECHANICAL PROBLEM,1
202,STALLED TRAILER TRUCK DUE TO MECHANICAL PROBLEM,1
201,STALLED BUS DUE TO MECHANICAL PROBLEM,1


## Step 7: Store Data in a SQL Database

To simulate a real-world data pipeline, the cleaned dataset is stored in a SQLite database.  
This allows us to perform structured queries and demonstrates how data can be integrated into a database environment.

In [11]:
import sqlite3

conn = sqlite3.connect("traffic.db")

df.to_sql("traffic_incidents", conn, if_exists="replace", index=False)

17312

## Step 8: Query the Data Using SQL

Using SQL queries, we extract aggregated statistics such as:
- incidents per city
- incidents per hour

These summarized datasets will later be used to build visualizations in Power BI.

In [12]:
query_city = """
SELECT city, COUNT(*) as accidents
FROM traffic_incidents
GROUP BY city
ORDER BY accidents DESC
"""

city_stats = pd.read_sql(query_city, conn)

city_stats.head()

,city,accidents
0,Quezon City,8725
1,Mandaluyong,2978
2,Makati City,2431
3,Pasig City,1723
4,Pasay City,445


In [13]:
query_hour = """
SELECT hour, COUNT(*) as accidents
FROM traffic_incidents
GROUP BY hour
ORDER BY hour
"""

hour_stats = pd.read_sql(query_hour, conn)

hour_stats

,hour,accidents
0,NaN,219
1,0.0,247
2,1.0,119
3,2.0,55
4,3.0,67
5,4.0,287
6,5.0,606
7,6.0,1013
8,7.0,1354
9,8.0,1271


## Step 9: Export Processed Data

Finally, the aggregated datasets are exported as CSV files.  
These files will be imported into Power BI to create interactive dashboards that visualize traffic incident patterns.

In [14]:
city_stats.to_csv("city_stats.csv", index=False)

hour_stats.to_csv("hour_stats.csv", index=False)

incident_types.to_csv("incident_types.csv", index=False)